# 03 — Clustering Analysis
### Global Airports ML Project
---
**Goal:** Group airports into meaningful clusters using KMeans and Hierarchical clustering.

## 1. Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

## 2. Load Processed Data & Scale

In [ ]:
from src.preprocessing import run_preprocessing_pipeline, scale_features, CLUSTER_FEATURES

df = run_preprocessing_pipeline('data/airports.csv')
X, scaler = scale_features(df, CLUSTER_FEATURES)

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {CLUSTER_FEATURES}')

## 3. Elbow Method

In [ ]:
from src.clustering import plot_elbow
plot_elbow(X, k_range=range(2, 11))

## 4. Silhouette Analysis

In [ ]:
from src.clustering import plot_silhouette
best_k = plot_silhouette(X, k_range=range(2, 8))
print(f'\nRecommended K: {best_k}')

## 5. KMeans Clustering (K=4)

In [ ]:
from src.clustering import kmeans_clustering

K = 4
labels, km_model, score = kmeans_clustering(X, n_clusters=K)
df['cluster'] = labels

print(f'\nCluster distribution:')
print(df['cluster'].value_counts().sort_index())

## 6. Cluster Profiles

In [ ]:
from src.clustering import cluster_profile

profile = cluster_profile(df, labels, feature_cols=CLUSTER_FEATURES)
profile

## 7. PCA Visualisation

In [ ]:
from src.clustering import plot_clusters_pca
plot_clusters_pca(X, labels, title=f'Airport Clusters K={K} (PCA 2D)')

## 8. Geographic Cluster Map

In [ ]:
from src.clustering import plot_clusters_geo
plot_clusters_geo(df, labels)

## 9. Silhouette Plot (Per Sample)

In [ ]:
from sklearn.metrics import silhouette_samples

sil_vals = silhouette_samples(X, labels)
y_lower = 10
fig, ax = plt.subplots(figsize=(9, 6))
colors = cm.tab10(np.linspace(0, 1, K))

for k, color in zip(range(K), colors):
    vals = np.sort(sil_vals[labels == k])
    size_k = vals.shape[0]
    y_upper = y_lower + size_k
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, facecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_k, str(k))
    y_lower = y_upper + 10

ax.axvline(x=score, color='red', linestyle='--', label=f'Avg score = {score:.3f}')
ax.set_title(f'Silhouette Plot — K={K}', fontsize=13)
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Hierarchical Clustering + Dendrogram

In [ ]:
linked = linkage(X, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked,
           labels=df['iata_code'].values,
           leaf_rotation=90,
           leaf_font_size=8,
           color_threshold=0.7 * max(linked[:, 2]))
plt.title('Hierarchical Clustering Dendrogram', fontsize=13)
plt.xlabel('Airport (IATA Code)')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 11. Sample Airports per Cluster

In [ ]:
for k in sorted(df['cluster'].unique()):
    print(f'\n━━━ Cluster {k} ━━━')
    sub = df[df['cluster'] == k][['name', 'country', 'passengers_millions', 'hub_type']]
    print(sub.to_string(index=False))

---
## Cluster Summary
| Cluster | Description |
|---|---|
| 0 | Mid-size regional hubs — moderate traffic |
| 1 | Large international hubs — high traffic |
| 2 | Smaller domestic airports — low traffic |
| 3 | High-altitude / specialised airports |

➡ Proceed to `04_classification_model.ipynb`